# Sprache, Emotion und Kognition

Das hier ist der Code mit dem ich unsere Stimuliliste zusammengestellt habe. Ich habe ihn (hoffentlich) weitestgehend so kommentiert bzw. dokumentiert, dass man auch ohne Programmierkenntnisse verstehen sollte, was genau ich hier mache.

Wenn irgendetwas unklar ist, dann schreibt mir bitte!

## Stimuli

Wir wollen 80 Wörter und 80 Nicht-Wörter, die den untenstehenden Charakteristika entsprechen sollen. Dafür lesen wir im Folgenden einige Datensätze aus und nutzen ihre Informationen dafür, Stück für Stück unsere Stimuliliste zusammenzubauen.

### Charakteristika - Wörter
- Sprache: Englisch
- Wortart: Nomen
- Länge*: Tendenz längere Wörter (min. 4/5 Buchstaben)
- Frequenz*: möglichst breite Frequenzspanne, möglichst gut verteilt (Ceiling Effect)
- Valenz*: nur neutrale Wörter
- Kognatenstatus: Nicht-Kognaten

### Charakteristika - Nichtwörter
- möglichst Länge wie die Wörter
- Wort könnte morphologisch ein Nomen des Englischen sein

> *Diese Charakteristika werden erst etwas weiter unten fest "eingestellt".

### Vorbereitungen

Hier importieren wir einige Module und stellen unsere kontinuierlichen Charakteristika Frequenz, Länge und Valenz ein.

In [ ]:
# pip install nltk
# pip install xlrd

  Using cached xlrd-2.0.2-py2.py3-none-any.whl.metadata (3.5 kB)
Using cached xlrd-2.0.2-py2.py3-none-any.whl (96 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Imports
import json
import numpy as np
import pandas as pd
import nltk
from nltk import pos_tag
from nltk.corpus import wordnet as wn

In [5]:
# Hier können wir einstellen wie lang, frequent oder valent unsere Wörter sein sollen

# Frequenz: die Range geht von 0.48 bis 6.33 (Logarithmus)
MIN_FREQ = 1        # nichts zu niedrig frequentes (Zipf?)
MAX_FREQ = 4       # nichts zu hochfrequentes

# Länge: die Range geht von 1.00 bis 28.00
MIN_LENGTH = 5
MAX_LENGTH = 8

# Valenz: die Range geht von 1.26 bis 8.53
# wobei laut der Originalstudie 1 = happy und 9 = unhappy
# Idee: Davon ist die Mitte genau 4. Neutral sollte also am Besten recht eng gefasst sein: 3.5 bis 4.5
MIN_VALENCE = 3.5
MAX_VALENCE = 4.5

## Wörter

### Datensätze der Wörter

Wir werden im Folgenden einige Datensätze einlesen. Hier ist ein Überblick:
- Frequenz - SUBTLEX (für verschiedene Sprachen verfügbar): https://www.ugent.be/pp/experimentele-psychologie/en/research/documents
- Valenz (EN) - Warriner et al. (2013): https://doi.org/10.3758/s13428-012-0314-x
- Kognaten - CogNet https://github.com/kbatsuren/CogNet und https://www.researchgate.net/publication/335389593_CogNet_a_Large-Scale_Cognate_Database

Für Wortarten benutzen wir zwei Methoden:
- da ein Großteil der Wörter nicht Nomen sein werden, filtern wir diese erst einmal grob heraus mit der ``pos_tag`` Methode von NLTK (Natural Language Toolkit). Die ist nicht ganz so akkurat, reicht aber zum groben Filtern, um den Datensatz zu verkleinern (und uns damit etwas Arbeit zu ersparen)
- später benutzen wir dann ein weit robustere Methode, den ''wordnet'' Korpus. Das ist aber ein bisschen aufwändiger, deswegen machen wir das erst ganz am Ende, wenn nur noch wenige Wörter übrig sind (je mehr Wörter wir überprüfen müssen, desto aufwänidger würde eine eh schon aufwändige Methode)

#### Idee
Wir fangen groß an (ein kompletter Datensatz) und sortieren nach und nach Wörter aus, die nicht zu unseren Charakteristika passen. Man kann es sich so vorstellen, dass wir jedes einzelne Wort dafür genau anschauen müssen, um zu überprüfen ob es passt oder nicht. Deswegen machen wir unser Aussortieren auch in einer bestimmten Reihenfolge, da manche Sachen leichter zu überprüfen sind (d.h., die können wir auch mal schnell bei ein paar tausend Wörtern anschauen) und manche Sachen schwerer (d.h., das machen wir lieber nur bei ein paar hunderten, weil es sonst zu lange dauert).

Und generell gilt natürlich: Methoden die schnell viele Wörter aussortieren (wie beispielsweise Wortarten) sollten wir so früh wie möglich anwenden, damit wir Wörter, die eh nicht passen, nicht unnötig ewig mit "rumschleppen".

#### Frequenz

Wir beginnen mit dem SUBTLEX Korpus. Den lesen wir ein und übernehmen ihn erst einmal komplett. Irgendwo müssen unsere Wörter ja herkommen.

In [15]:
# einlesen
df = pd.read_excel("SUBTLEXusfrequencyabove1.xls")

# Wie lang ist der Datensatz?
original_size = len(df)

# Feedback: 
print(f"Read SUBTLex dataset, found {original_size} words.")

Read SUBTLex dataset, found 60384 words.


In [13]:
# So sieht dieser Datensatz zum Beispiel aus:

# 	Word	FREQcount	CDcount	FREQlow	Cdlow	SUBTLWF	Lg10WF	SUBTLCD	Lg10CD
# 0	the	1501908	8388	1339811	8388	29449.176471	6.176644	100.000000	3.923710
# 1	to	1156570	8383	1138435	8380	22677.843137	6.063172	99.940391	3.923451
# 2	a	1041179	8382	976941	8380	20415.274510	6.017526	99.928469	3.923399
# 3	you	2134713	8381	1595028	8376	41857.117647	6.329340	99.916547	3.923348
# 4	and	682780	8379	515365	8374	13387.843137	5.834281	99.892704	3.923244

In [ ]:
# jetzt können wir den Datensatz im Hinblick auf Frequenz vorverarbeiten

# Damit wir nicht aus Versehen was kaputt machen: nimm dir nur die zwei Spalten die wir wirklich brauchen und speichere sie in einem neuen Datensatz
# "Word" -> Wörter
# "Lg10WF" -> Log10 Häufigkeit (warum Log10? ist meistens bisschen stabiler mit logarithmierten Zahlen zu rechnen als mit normalen gezählten Häufigkeiten)
freq_df = df[["Word", "Lg10WF"]]

# Wir werden ab jetzt mit diesem Datensatz weiterarbeiten, deswegen benennen wir die Spalten erstmal um in etwas was wir uns besser merken können
freq_df = freq_df.rename(columns={"Word": "word"})
freq_df = freq_df.rename(columns={"Lg10WF": "freq"})

# Jetzt kann es sein, dass ein und dasselbe Wort in verschiedenen Schreibweisen in dem Datensatz auftaucht, z.B. "Mouse" und "mouse"
# Wenn wir die Wörter später im Experiment benutzen, werden die aber ja komplett großgeschrieben dargestellt
# Das heißt, wir normalisieren jetzt einmal alle Wörter
# (= alle werden jetzt gleich geschrieben - allerdings klein, weil wir später noch einen Wortarten Tagger über die Wörter laufen lassen und der mag 
# kleingeschriebene Wörter ein bisschen lieber als großgeschriebene):
freq_df["word"] = freq_df["word"].str.lower()

# Und dann nehmen wir von den Einzelhäufigkeiten der verschiedenen Schreibweisen die höchste (das Maximum)
# Bsp.: "Mouse": 1.5; "mouse": 3.3 wird zu "mouse": 3.3
# wir nehmen hier an, dass das eine gute Annäherung ist, vor allem aber:
# das ist - zumindest für unseren Zweck - auch das fairste & einfachste, da wir mit einem Logarithmus arbeiten
# Addieren der Einzelhäufigkeiten wäre daher zum Beipiel keine gute Lösung
freq_df = freq_df.groupby("word", as_index=False).agg({"freq": max})

# Feedback:
print(f"""Preprocessed frequency, now we have {len(freq_df)} words.
      This means that we have already decluttered {original_size - len(freq_df)} words""")

# So sieht das dann aus
freq_df

Preprocessed frequency, now we have 60382 words.
      This means that we have already decluttered 2 words


,word,freq
0,a,6.017526
1,aa,1.944483
2,aaa,1.414973
3,aah,3.429591
4,aahing,0.477121
...,...,...
60377,zwieback,0.477121
60378,zydeco,1.176091
60379,zygoma,0.477121
60380,zygomatic,0.778151


In [17]:
# Jetzt können wir den Datensatz anhand unserer oben spezifizierten Frequenzkriterien kürzen
# Alles was außerhalb der Range liegt, brauchen wir nicht

# Nimm dir nur die Zeilen die größer gleich unserer Mindesthäufigkeit und kleiner gleich unserer Maximalhäufigkeit liegen
freq_range_df = freq_df[(freq_df["freq"] >= MIN_FREQ) & (freq_df["freq"] <= MAX_FREQ)]

# Feedback
print (f"""Found {len(freq_range_df)} words in our desired range from {MIN_FREQ} to {MAX_FREQ}.
       The highest observed frequency is {freq_range_df["freq"].max():.2f} and the lowest is {freq_range_df["freq"].min():.2f}.
       Mean: {freq_range_df["freq"].mean():.2f}, Standard deviation: {freq_range_df["freq"].std():.2f}.
       We have already decluttered {original_size - len(freq_range_df)} words""")

# So sieht das dann aus
freq_range_df

Found 37176 words in our desired range from 1 to 4.
       The highest observed frequency is 4.00 and the lowest is 1.00.
       Mean: 1.79, Standard deviation: 0.64.
       We have already decluttered 23208 words


,word,freq
1,aa,1.944483
2,aaa,1.414973
3,aah,3.429591
6,aardvark,1.342423
7,aargh,1.531479
...,...,...
60370,zs,1.041393
60371,zucchini,1.698970
60373,zulu,1.707570
60376,zurich,2.096910


#### Wortart

Über die Wortart werden wir wahrscheinlich die meisten unserer oben gesammelten Wörter wieder aussortieren. Deswegen ist es gut, wenn wir das direkt hier einmal grob machen, weil unser Datensatz dann direkt ein bisschen kleiner ist und wir weniger verarbeiten müssen (= wir brauchen weniger Ressourcen).

Dafür benutzen wir die Methode ``pos_tag``, das ist ein Wortarten-Tagset von der sogenannten Penn Treebank:

Hier gibt es verschiedene Abstufungen von Nomen, wir interessieren uns aber nur für Nomen im Singular, nicht im Plural und auch nicht für Eigennamen \
$\to$ siehe "NN" in der folgenden Übersicht

In [ ]:
# nltk.help.upenn_tagset()

# NN: noun, common, singular or mass
#     common-carrier cabbage knuckle-duster Casino afghan shed thermostat
#     investment slide humour falloff slick wind hyena override subhumanity
#     machinist ...
# NNP: noun, proper, singular
#     Motown Venneboerger Czestochwa Ranzer Conchita Trumplane Christos
#     Oceanside Escobar Kreisler Sawyer Cougar Yvette Ervin ODI Darryl CTCA
#     Shannon A.K.C. Meltex Liverpool ...
# NNPS: noun, proper, plural
#     Americans Americas Amharas Amityvilles Amusements Anarcho-Syndicalists
#     Andalusians Andes Andruses Angels Animals Anthony Antilles Antiques
#     Apache Apaches Apocrypha ...
# NNS: noun, common, plural
#     undergraduates scotches bric-a-brac products bodyguards facets coasts
#     divestitures storehouses designs clubs fragrances averages
#     subjectivists apprehensions muses factory-jobs ...

In [20]:
# wir kopieren den Datensatz um nicht aus Versehen was kaputt zu machen
tag_df = freq_range_df.copy()

# dann sammeln wir alle Wörter aus unserem Datensatz in einer Liste
words = tag_df["word"].tolist()

# lassen den Tagger über die Liste laufen
tagged_words = pos_tag(words)

# und hängen die Tags an unseren Datensatz an
tag_df["pos"] = [tag for _, tag in tagged_words]

# So sieht das dann aus
tag_df

,word,freq,pos
1,aa,1.944483,NN
2,aaa,1.414973,NN
3,aah,3.429591,NN
6,aardvark,1.342423,NN
7,aargh,1.531479,NN
...,...,...,...
60370,zs,1.041393,NN
60371,zucchini,1.698970,NNP
60373,zulu,1.707570,NNP
60376,zurich,2.096910,NNP


In [21]:
# Jetzt können wir den Datensatz wieder kürzen
# Alles was kein Nomen ist, brauchen wir nicht

# Nimm dir nur die Zeilen mit der Wortart NN (Nomen)
noun_tag_df = tag_df[tag_df["pos"] == "NN"]

# Feedback
print (f"""Found {len(noun_tag_df)} words after filtering for nouns.
       We have already decluttered {original_size - len(noun_tag_df)} words""")

# So sieht das dann aus
noun_tag_df

Found 11792 words after filtering for nouns.
       We have already decluttered 48592 words


,word,freq,pos
1,aa,1.944483,NN
2,aaa,1.414973,NN
3,aah,3.429591,NN
6,aardvark,1.342423,NN
7,aargh,1.531479,NN
...,...,...,...
60365,zoos,1.602060,NN
60367,zorro,2.136721,NN
60369,zowie,1.278754,NN
60370,zs,1.041393,NN


#### Länge

Jetzt machen wir im Grunde das Gleiche was wir mit der Frequenz und den Wortarten vorher gemacht haben mit der Wortlänge. Für jedes Wort zählen wir die Buchstaben und hängen sie an den Datensatz ran.

In [22]:
# Kopieren um nichts kaputt zu machen
len_df = noun_tag_df.copy()

# Länge anhängen
len_df["length"] = len_df["word"].str.len()

# So sieht das dann aus
len_df

,word,freq,pos,length
1,aa,1.944483,NN,2
2,aaa,1.414973,NN,3
3,aah,3.429591,NN,3
6,aardvark,1.342423,NN,8
7,aargh,1.531479,NN,5
...,...,...,...,...
60365,zoos,1.602060,NN,4
60367,zorro,2.136721,NN,5
60369,zowie,1.278754,NN,5
60370,zs,1.041393,NN,2


In [23]:
# Kürzen anhand unserer Range von oben
len_range_df = len_df[(len_df["length"] >= MIN_LENGTH) & (len_df["length"] <= MAX_LENGTH)]

# Feedback
print (f"""Found {len(len_range_df)} words in our desired range from {MIN_LENGTH} to {MAX_LENGTH}.
       The highest observed length is {len_range_df["length"].max():.2f} and the lowest is {len_range_df["length"].min():.2f}.
       Mean: {len_range_df["length"].mean():.2f}, Standard deviation: {len_range_df["length"].std():.2f}.
       We have already decluttered {original_size - len(len_range_df)} words""")

# So sieht das dann aus
len_range_df

Found 6813 words in our desired range from 5 to 8.
       The highest observed length is 8.00 and the lowest is 5.00.
       Mean: 6.48, Standard deviation: 1.07.
       We have already decluttered 53571 words


,word,freq,pos,length
6,aardvark,1.342423,NN,8
7,aargh,1.531479,NN,5
8,aaron,2.873902,NN,5
12,aback,1.204120,NN,5
13,abacus,1.113943,NN,6
...,...,...,...,...
60334,zirconia,1.113943,NN,8
60359,zoology,1.255273,NN,7
60367,zorro,2.136721,NN,5
60369,zowie,1.278754,NN,5


#### Valenz

Ähnlich gehen wir auch bei der Valenz vor. Da wir aber noch keine Werte für Valenz haben, müssen wir zunächst erst einen weiteren Datensatz einlesen, in welchem entsprechende Werte gespeichert sind.

In [24]:
# Valenzdatensatz einlesen
valence_df = pd.read_csv("BRM-emot-submit.csv")

# Wir nehmen nur die Spalten die uns interessieren
# "Word": Wort
# "V.Mean.Sum": Durchschnittsvalenz (von Valenzratings aus einer experimentellen Studie)
valence_df = valence_df[["Word", "V.Mean.Sum"]]

# Und wieder umbenennen in etwas was wir uns besser merken können
valence_df = valence_df.rename(columns={"Word": "word"})
valence_df = valence_df.rename(columns={"V.Mean.Sum": "valence"})

# Feedback
print (f"""Found {len(valence_df)} words in the valence dataset.
       The highest possible valence is {valence_df["valence"].max():.2f} and the lowest is {valence_df["valence"].min():.2f}.""")

# So sieht das dann aus
valence_df

Found 13915 words in the valence dataset.
       The highest possible valence is 8.53 and the lowest is 1.26.


,word,valence
0,aardvark,6.26
1,abalone,5.30
2,abandon,2.84
3,abandonment,2.63
4,abbey,5.85
...,...,...
13910,zone,4.75
13911,zoning,4.65
13912,zoo,7.00
13913,zoom,5.86


In [25]:
# Auch hier dürfen wir wieder normalisieren, d.h. alles kleinschreiben
# (schließlich müssen wir die Wörter aus diesem Datensatz gleich mit unseren Wörtern von oben abgleichen und die sind ja auch normalisiert)
valence_df["word"] = valence_df["word"].str.lower()

# Und falls wir ein und dasselbe Wort in verschiedenen Schreibweisen haben (siehe Mouse - mouse Beispiel von oben),
# nehmen wir in solchen Fällen die Durchschnittsvalenz aller Schreibweisen
mean_valence_df = valence_df.groupby("word", as_index=False)["valence"].mean()

# Feedback
print (f"""Found {len(mean_valence_df)} words in the valence dataset after normalizing words.
       The highest possible valence is {mean_valence_df["valence"].max():.2f} and the lowest is {mean_valence_df["valence"].min():.2f}.
       This means that {len(valence_df) - len(mean_valence_df)} words were collapsed.""")

# So sieht das dann aus
mean_valence_df

Found 13904 words in the valence dataset after normalizing words.
       The highest possible valence is 8.53 and the lowest is 1.26.
       This means that 11 words were collapsed.


,word,valence
0,aardvark,6.26
1,abalone,5.30
2,abandon,2.84
3,abandonment,2.63
4,abbey,5.85
...,...,...
13899,zone,4.75
13900,zoning,4.65
13901,zoo,7.00
13902,zoom,5.86


In [ ]:
# Gleich können wir den Datensatz wieder anhand unserer spezifizierten Range kürzen
# Dafür müssen wir allerdings zuerst den soeben eingelesen Datensatz auf unseren Datensatz von oben abbilden

# Kopiere den Datensatz von oben
val_df = len_range_df.copy()

# Bilde den neuen Valenzdatensatz auf unseren Datensatz ab
# Wir hängen praktisch einfach eine neue Spalte mit Valenzen an
# Wenn ein Wort in unserem Datensatz vorhanden ist, aber nicht in dem Valenzdatensatz, dann wird die Valenz als "NaN" gezählt
val_df = val_df.merge(mean_valence_df, on= "word", how = "left")

print("Done")

# So sieht das dann aus
val_df

Done


,word,freq,pos,length,valence
0,aardvark,1.342423,NN,8,6.26
1,aargh,1.531479,NN,5,NaN
2,aaron,2.873902,NN,5,NaN
3,aback,1.204120,NN,5,NaN
4,abacus,1.113943,NN,6,NaN
...,...,...,...,...,...
6808,zirconia,1.113943,NN,8,NaN
6809,zoology,1.255273,NN,7,NaN
6810,zorro,2.136721,NN,5,NaN
6811,zowie,1.278754,NN,5,NaN


In [ ]:
# Jetzt kürzen wir
# Alles was außerhald der oben spezifizierten Range ist fliegt raus
# Sowie auch alle Wörter bei denen die Valenz unbekannt ist (NaNs)
val_range_df = val_df[(val_df["valence"] >= MIN_VALENCE) & (val_df["valence"] <= MAX_VALENCE) & (val_df["valence"].notna())]

# Feedback
print (f"""Found {len(val_range_df)} words in our desired range: {MIN_VALENCE} - {MAX_VALENCE}.
       The highest valence is {val_range_df["valence"].max():.2f} and the lowest is {val_range_df["valence"].min():.2f}.
       Mean: {val_range_df["valence"].mean():.2f}, Standard deviation: {val_range_df["valence"].std():.2f}.
       We have already decluttered {original_size - len(val_range_df)} words""")

# So sieht das dann aus
val_range_df

Found 645 words in our desired range: 3.5 - 4.5.
       The highest valence is 4.50 and the lowest is 3.50.
       Mean: 4.04, Standard deviation: 0.30.
       We have already decluttered 59739 words


,word,freq,pos,length,valence
20,absence,2.509203,NN,7,3.86
24,abstain,1.431364,NN,7,4.40
27,abyss,1.959041,NN,5,3.90
49,adder,1.869232,NN,5,4.45
57,adjuster,1.322219,NN,8,4.25
...,...,...,...,...,...
6750,woozy,1.643453,NN,5,3.90
6763,wreak,1.491362,NN,5,3.50
6765,wrecker,1.431364,NN,7,4.05
6767,wrestler,1.963788,NN,8,4.26


#### Nomen #2

Der erste Wortartenfilter mit ``pos_tag`` war nur eine grobe Vorauswahl. Nun überprüfen wir die verbleibenden Wörter mit ``WordNet``, einer deutlich umfangreicheren linguistischen Datenbank. Dadurch können wir zuverlässiger einschätzen, ob ein Wort tatsächlich hauptsächlich als Nomen verwendet wird.

WordNet speichert für viele Wörter verschiedene Bedeutungen ("Synsets"). Manche Wörter kommen als Nomen, Verben oder Adjektive vor. Wir zählen daher, wie häufig ein Wort als Nomen im Vergleich zu anderen Wortarten vertreten ist. Daraus berechnen wir einen Wert zwischen 0 und 1:

- 1 = praktisch ausschließlich Nomen
- 0 = praktisch nie Nomen

In [ ]:


def noun_score(word):
    noun_synsets = wn.synsets(word, pos=wn.NOUN)
    verb_synsets = wn.synsets(word, pos=wn.VERB)
    adj_synsets  = wn.synsets(word, pos=wn.ADJ)

    noun_count = len(noun_synsets)
    other_count = len(verb_synsets) + len(adj_synsets)

    return noun_count / (noun_count + other_count + 1e-9)

In [ ]:
# safer
# val_range_df = val_range_df.copy()
# val_range_df["noun_score"] = ...


val_range_df["noun_score"] = val_range_df["word"].apply(noun_score)

noun_df = val_range_df[val_range_df["noun_score"] >= 0.7]

In [389]:
len(noun_df)

330

#### Kognaten

Hier sortieren wir einfach alle Kognaten aus. Mithilfe der CogNet-Datenbank suchen wir deshalb nach englischen Wörtern, die als deutsche Kognaten markiert sind, und entfernen diese.

Das Durchsuchen der gesamten CogNet-Datenbank dauert mehrere Minuten. Deshalb speichern wir die gefundenen Kognaten einmalig in einer Datei. Bei späteren Durchläufen können wir die Datei direkt laden und müssen die Berechnung nicht erneut durchführen.

In [390]:
# Datensätze einlesen




# Cognates
# CogNet-v1.0.tsv
cognate_df = pd.read_csv("CogNet-v1.0.tsv", sep="\t")        # cognate status

In [391]:
# big dataset -> running this will take some time, so we save it in a file and if the file exists we dont recompute



# cognate_set = set()

# for _, r in cognate_df.iterrows():
#     w1, l1 = r["word 1"], r["lang 1"]   # e.g. w1: "cat", l1: "eng", w2: "Katze", l2: "deu"
#     w2, l2 = r["word 2"], r["lang 2"]

#     if l1 == "eng" and l2 == "deu":
#         cognate_set.add(str(w1))
#     if l2 == "eng" and l1 == "deu":
#         cognate_set.add(str(w2))



cache_file = "cognate_set.json"

# -----------------------------
# Load if exists
# -----------------------------
try:
    with open(cache_file, "r") as f:
        cognate_set = set(json.load(f))
    print(f"Loaded cognate_set with {len(cognate_set)} items")

# -----------------------------
# Compute if not exists
# -----------------------------
except FileNotFoundError:
    cognate_set = set()

    for _, r in cognate_df.iterrows():
        w1, l1 = r["word 1"], r["lang 1"]
        w2, l2 = r["word 2"], r["lang 2"]

        if l1 == "eng" and l2 == "deu":
            cognate_set.add(str(w1))
        elif l2 == "eng" and l1 == "deu":
            cognate_set.add(str(w2))

    with open(cache_file, "w") as f:
        json.dump(list(cognate_set), f)

    print(f"Computed and saved cognate_set with {len(cognate_set)} items")

Loaded cognate_set with 7196 items


In [392]:
# run to see our cognate set
cognate_set

{'european',
 'diaphanous',
 'scandal',
 'overrate',
 'dextrality',
 'hydrology',
 'realize',
 'hydrocephaly',
 'spritz',
 'katzenjammer',
 'html',
 'adelie',
 'bathyscape',
 'incubator',
 'synchronism',
 'catholicism',
 'gerfalcon',
 'hedonic',
 'irony',
 'infantryman',
 'outer planet',
 'peanut butter',
 'cork',
 'originality',
 'tactical',
 'destructive',
 'milkshake',
 'grade',
 'sociolinguistic',
 'cedar',
 'blog',
 'buffalo',
 'therapy',
 'insignia',
 'financial backing',
 'laminate',
 'transcendental number',
 'fluctuate',
 'kafkaesque',
 'waken',
 'livery',
 'calligraphy',
 'tigress',
 'antihistamine',
 'allegorically',
 'ghetto',
 'paranoid',
 'croatian',
 'senile',
 'adverbial',
 'kopek',
 'cystitis',
 'radar',
 'specialiser',
 'lustfully',
 'histologic',
 'futurism',
 'inaugural',
 'pistachio',
 'kelvin',
 'leucocyte',
 'soap opera',
 'frontal lobe',
 'order',
 'eccentricity',
 'biblical',
 'aristotelianism',
 'tangential',
 'plus sign',
 'disco',
 'motorcycle',
 'memoranda'

In [394]:
non_cognate_df = noun_df.copy()
non_cognate_df = non_cognate_df[~non_cognate_df["word"].isin(cognate_set)]



# Feedback
print (f"""Found {len(non_cognate_df)} words that are not cognates.
       We have already decluttered {original_size - len(non_cognate_df)} words""")

non_cognate_df

Found 231 words that are not cognates.
       We have already decluttered 60153 words


,word,freq,pos,length,valence,noun_score
27,abyss,1.959041,NN,5,3.90,1.0
49,adder,1.869232,NN,5,4.45,1.0
57,adjuster,1.322219,NN,8,4.25,1.0
114,alderman,1.623249,NN,8,4.44,1.0
133,alley,2.920123,NN,5,4.17,1.0
...,...,...,...,...,...,...
6640,weaponry,1.740363,NN,8,3.71,1.0
6646,weenie,2.161368,NN,6,3.86,1.0
6653,welfare,2.605305,NN,7,3.58,1.0
6765,wrecker,1.431364,NN,7,4.05,1.0


In [395]:
# Final feedback
print (f"""Found {len(non_cognate_df)} words that fit our prerequisites.
       In total, we have decluttered {original_size - len(non_cognate_df)} words""")

Found 231 words that fit our prerequisites.
       In total, we have decluttered 60153 words


### Uniforme Verteilung der Frequenzen

Explain:

Why random sampling is not enough.
Why you want words across the whole frequency range.
Why evenly spaced target frequencies are generated.

In [ ]:
df = non_cognate_df
# in the following code - TODO anpassen 

In [ ]:
## TODO:
# control frequency distribution (similar distribution over the full range)
# add nonwords to the dataset (make sure they have a similar distribution of lengths)
# statistical testing
# documentation & comments

In [ ]:
# (kind of) uniform frequency distirbution

# ideally, we want to selected words to be uniformly distributed bteween MIN_FREQ and MAX_FREQ

# so what we do is:
# generate evenly spaced target frequencies and pick the closest word to each target

In [ ]:

N = 100 # should be 80 in the end, so we want some space to declutter unfitting words


targets = np.linspace(df["freq"].min(), df["freq"].max(), N)

remaining = df.copy()
selected = []

for target in targets:
    idx = (remaining["freq"] - target).abs().idxmin()
    selected.append(remaining.loc[idx])
    remaining = remaining.drop(idx)

sampled = pd.DataFrame(selected)

# non_cognate_df

In [ ]:
# this prduces a subset whose frequency distriution is approximately uniform across the entire frequency range, i.e., (kind of) equally distributed among frequencies

In [ ]:
# plot

In [ ]:
# histogram comparison

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.hist(df["freq"], bins=30, alpha=0.5, label="Original")
plt.hist(sampled["freq"], bins=30, alpha=0.5, label="Sampled")

plt.xlabel("Frequency")
plt.ylabel("Number of words")
plt.title("Frequency Distribution")
plt.legend()

plt.show()

In [ ]:
# density comparison

plt.figure(figsize=(10, 5))

df["freq"].plot(kind="density", label="Original")
sampled["freq"].plot(kind="density", label="Sampled")

plt.xlabel("Frequency")
plt.title("Frequency Density")
plt.legend()

plt.show()

In [ ]:
# check uniformity directly

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.scatter(
    range(len(sampled)),
    sampled["freq"].sort_values(),
    s=10
)

plt.xlabel("Selected word rank")
plt.ylabel("Frequency")
plt.title("Frequencies of Selected Words")

plt.show()

In [ ]:
# inspect coverage numerically

print("Original:")
print(df["freq"].describe())

print("\nSampled:")
print(sampled["freq"].describe())

## Nicht-Wörter


Non-word section needs much more documentation

This is currently the least documented part.

You should explain:

Data source

Where the non-words come from.

Why length matching is important

Something like:

Damit Unterschiede zwischen Wörtern und Nicht-Wörtern nicht einfach durch die Wortlänge erklärt werden können, wählen wir Nicht-Wörter mit derselben Längenverteilung wie unsere Wörter aus.


Explain:

count word lengths,
determine how many non-words of each length are needed,
randomly sample exactly that many.

In [ ]:
## 

In [ ]:
# Nicht-Wörter
    # Wuggy (Keuleers & Brysbaert, 2010) | UniPseudo (New et al., 2024)
        # generieren sprachspezifische Pseudowörter mit kontrollierten Eigenschaften
    # ARC Nonword Database (Rastle et al., 2002)
        # Datenbank für englische Nonwords

## Letztes Mal benutzt:
# ARC Nonword Database

In [ ]:
# # ARC
# min länge 5
# max länge 8

#  only orthographically existing onsets
#  only orthographically existing bodies
# only legal bigrams

# http://www.cogsci.mq.edu.au/research/resources/nwdb/nwdb.html

# http://www.cogsci.mq.edu.au/cgi-bin/nwsrch.cgi

NameError: name 'MAX_LENGTH' is not defined

In [ ]:
# read file to dataframe



# nonwords_original_results_clean.txt (deleted header from the google drive file)

# structure: word(N) tab idx(PRON) tab length(NL)


# W               PRON            NL   

In [ ]:
matched_nonwords = (
    sampled["length"]
    .value_counts()
    .sort_index()
)

selected_nonwords = []

for length, n_needed in matched_nonwords.items():
    candidates = nonwords[nonwords["length"] == length]

    if len(candidates) < n_needed:
        raise ValueError(
            f"Need {n_needed} nonwords of length {length}, "
            f"but only {len(candidates)} available."
        )

    selected_nonwords.append(
        candidates.sample(n=n_needed, random_state=42)
    )

selected_nonwords = pd.concat(selected_nonwords).reset_index(drop=True)

In [ ]:
# save
non_cognate_df.to_csv("dataframe_exp.csv", encoding='utf-8', index=False, sep = ";")
print(f"Done! The dataset was saved to \"dataframe_exp.csv\". Check your folder :)")

In [ ]:
   



# Liste schicken & sagen was sie anschauen sollen

# trial nummer in model mit aufnehmen für ermüdungseffekte untersuchen

# aufbauende Gruppe, negative Gruppe bei richtigen und falschen Antworten

# oder feedback vs kein feedback

# wie wirkt sich feedback auf die performance aus und hat das auswirkungen auf die persönlichkeitsmerkmale

# alter flexibel

